# 03 — Fine-tuned RoBERTa

**Goal:** Fine-tune `roberta-base` to predict song topic (8 classes) from cleaned lyrics.

**Depends on:** `data/processed/train.csv` and `data/processed/test.csv` produced by `00_preprocessing.ipynb`.

> Run this notebook in **Google Colab with a GPU runtime**:  
> Runtime → Change runtime type → Hardware accelerator → GPU

## 0. Colab setup - install packages and check GPU

In [1]:
# Install / upgrade required packages (safe to re-run)
!pip install transformers accelerate -q

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected. '
          'Go to Runtime -> Change runtime type →->GPU before training.')
else:
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 1. Imports and paths

In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
from torch.utils.data import Dataset
from transformers import (RobertaTokenizer, RobertaForSequenceClassification,
                          TrainingArguments, Trainer)

# Upload train.csv and test.csv from your local machine
from google.colab import files
print("Upload train.csv and test.csv")
uploaded = files.upload()

DATA_DIR = '/content'
RESULTS_DIR = '/content/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print('DATA_DIR   :', DATA_DIR)
print('RESULTS_DIR:', RESULTS_DIR)

Upload train.csv and test.csv


Saving test.csv to test.csv
Saving topic_distribution.png to topic_distribution.png
Saving topic_label_map.json to topic_label_map.json
Saving train.csv to train.csv
DATA_DIR   : /content
RESULTS_DIR: /content/results


## 2. Load data

In [4]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

# Use lyrics_clean as input; topic_label (0–7) as target
X_train = train_df['lyrics_clean'].fillna('').tolist()
y_train = train_df['topic_label'].tolist()
X_test  = test_df['lyrics_clean'].fillna('').tolist()
y_test  = test_df['topic_label'].tolist()

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Classes: {train_df["topic_label"].nunique()}')
print(train_df['topic'].value_counts().sort_index().to_string())

Train: 22,233  |  Test: 5,559
Classes: 8
topic
feelings       463
music         1784
night/time    1414
obscene       3864
romantic      1181
sadness       4788
violence      4495
world/life    4244


## 3. Label mapping

In [5]:
# LabelEncoder in 00_preprocessing.ipynb encodes topics alphabetically
label_map = {
    0: 'feelings',
    1: 'music',
    2: 'night/time',
    3: 'obscene',
    4: 'romantic',
    5: 'sadness',
    6: 'violence',
    7: 'world/life',
}
TOPIC_NAMES = [label_map[i] for i in range(len(label_map))]
N_CLASSES   = len(TOPIC_NAMES)

print('Label mapping:')
for k, v in label_map.items():
    print(f'  {k} → {v}')

Label mapping:
  0 → feelings
  1 → music
  2 → night/time
  3 → obscene
  4 → romantic
  5 → sadness
  6 → violence
  7 → world/life


## 4. Tokenisation and PyTorch Dataset

In [6]:
MAX_LENGTH = 128

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

class LyricsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt',
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = LyricsDataset(X_train, y_train, tokenizer, MAX_LENGTH)
test_dataset  = LyricsDataset(X_test,  y_test,  tokenizer, MAX_LENGTH)

print(f'Train dataset: {len(train_dataset):,} samples')
print(f'Test dataset : {len(test_dataset):,} samples')
print(f'Input IDs shape (first sample): {train_dataset[0]["input_ids"].shape}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train dataset: 22,233 samples
Test dataset : 5,559 samples
Input IDs shape (first sample): torch.Size([128])


## 5. Load model

In [7]:
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=N_CLASSES,
)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable:,}')

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters    : 124,651,784
Trainable parameters: 124,651,784


## 6. Training

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'macro_f1': float(f1_score(labels, preds, average='macro')),
    }

In [9]:
training_args = TrainingArguments(
    output_dir='./roberta-topic-checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_macro_f1',
    greater_is_better=True,
    logging_dir='./logs',
    logging_steps=100,
    report_to='none',
    fp16=torch.cuda.is_available(),  # mixed precision on GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.363349,0.323793,0.892427,0.889980
2,0.253757,0.276426,0.916172,0.913788
3,0.134118,0.305994,0.922468,0.917995


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=4170, training_loss=0.3088819512932135, metrics={'train_runtime': 606.0011, 'train_samples_per_second': 110.064, 'train_steps_per_second': 6.881, 'total_flos': 4387547421739008.0, 'train_loss': 0.3088819512932135, 'epoch': 3.0})

## 7. Evaluate on test set

In [10]:
preds_output = trainer.predict(test_dataset)
y_pred       = np.argmax(preds_output.predictions, axis=-1)
y_true       = np.array(y_test)

macro_f1 = f1_score(y_true, y_pred, average='macro')
accuracy = accuracy_score(y_true, y_pred)

print(f'Accuracy : {accuracy:.4f}')
print(f'Macro F1 : {macro_f1:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=TOPIC_NAMES))

Accuracy : 0.9225
Macro F1 : 0.9180

              precision    recall  f1-score   support

    feelings       0.85      0.91      0.88       116
       music       0.91      0.94      0.93       446
  night/time       0.91      0.92      0.91       354
     obscene       0.95      0.93      0.94       966
    romantic       0.94      0.91      0.92       295
     sadness       0.92      0.93      0.92      1197
    violence       0.92      0.91      0.91      1124
  world/life       0.92      0.92      0.92      1061

    accuracy                           0.92      5559
   macro avg       0.91      0.92      0.92      5559
weighted avg       0.92      0.92      0.92      5559



## 8. Save predictions

In [11]:
preds_df = pd.DataFrame({
    'topic_label': y_true,
    'y_pred':      y_pred,
})
out_path = os.path.join(DATA_DIR, 'roberta_preds.csv')
preds_df.to_csv(out_path, index=False)
print(f'Saved {len(preds_df):,}')

Saved 5,559


## 9. Download predictions (Colab)

Run this cell to download `roberta_preds.csv` to your local machine so it can be placed in `data/` for `04_evaluation.ipynb`.

In [12]:
from google.colab import files
files.download(out_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>